# TP 3b — Jupyter et formats (corrigé)
Air'Cité : les mesures horaires de NO2, PM10 et température de 12 capteurs, du 2 au 29 mars 2026.

- **source** : `mesures.csv`, produit au TP 2b (copie de référence dans `../donnees/`)
- **unités** : µg/m³ pour NO2 et PM10, °C pour la température
- **noyau** : Python (aircite)

## 1. Le bon noyau
Le chemin de l'interpréteur doit pointer dans l'environnement `aircite` (ou `.venv`).

In [1]:
%run ../tp3a-environnement/verifier_env.py

Python      3.11.8
Interpréteur C:\Users\vous\Documents\aircite\.venv\Scripts\python.exe
Préfixe      C:\Users\vous\Documents\aircite\.venv
  OK  numpy       2.4.6


  OK  pandas      2.3.3


  OK  matplotlib  3.11.2
  OK  scipy       1.17.1
  OK  pyarrow     25.0.1


  OK  openpyxl    3.1.5
  OK  ipykernel   7.4.0

Environnement prêt.


## 2. IPython : se documenter et mesurer

In [2]:
import pandas as pd
pd.read_csv?

In [3]:
%timeit sum(range(100_000))

2.78 ms ± 386 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
import numpy as np
%timeit np.arange(100_000).sum()

60.8 μs ± 3.86 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 3. Lire `mesures.csv`, sans puis avec `parse_dates`

In [5]:
brut = pd.read_csv("../donnees/mesures.csv")
print(brut.shape)
brut.dtypes

(8064, 6)


horodatage     object
capteur        object
quartier       object
no2           float64
pm10          float64
temp          float64
dtype: object

In [6]:
m = pd.read_csv("../donnees/mesures.csv", parse_dates=["horodatage"])
m.dtypes

horodatage    datetime64[ns]
capteur               object
quartier              object
no2                  float64
pm10                 float64
temp                 float64
dtype: object

In [7]:
m.head(3)

,horodatage,capteur,quartier,no2,pm10,temp
0,2026-03-02 00:00:00,AQ-01,Centre,NaN,12.1,5.5
1,2026-03-02 01:00:00,AQ-01,Centre,11.3,11.3,4.4
2,2026-03-02 02:00:00,AQ-01,Centre,24.5,17.0,1.7


## 4. Écrire dans trois autres formats, relire, comparer

In [8]:
m.to_parquet("mesures.parquet")
m.to_excel("mesures.xlsx", index=False)
m.to_json("mesures.json", orient="records", lines=True)

In [9]:
import os
lus = {
    "CSV": pd.read_csv("../donnees/mesures.csv"),
    "Parquet": pd.read_parquet("mesures.parquet"),
    "Excel": pd.read_excel("mesures.xlsx"),
    "JSON": pd.read_json("mesures.json", lines=True),
}
fichiers = {"CSV": "../donnees/mesures.csv", "Parquet": "mesures.parquet",
            "Excel": "mesures.xlsx", "JSON": "mesures.json"}
pd.DataFrame({
    "taille (Ko)": {f: round(os.path.getsize(p) / 1024) for f, p in fichiers.items()},
    "type de horodatage": {f: str(d["horodatage"].dtype) for f, d in lus.items()},
    "NO2 manquants": {f: int(d["no2"].isna().sum()) for f, d in lus.items()},
})

,taille (Ko),type de horodatage,NO2 manquants
CSV,382,object,151
Parquet,42,datetime64[ns],151
Excel,246,datetime64[ns],151
JSON,808,int64,151


Chaque format a sa façon de traiter l'horodatage :
- **CSV** : il revient en texte (`object`), il faut `parse_dates` à chaque lecture ;
- **Parquet** : il garde le type, et c'est de loin le plus petit (≈ 9 fois moins que le CSV) ;
- **Excel** : il garde le type, mais c'est le plus lent à relire ;
- **JSON** : il revient en entier (`int64`), des millisecondes depuis 1970.

## 5. Avant de partager
Menu **Kernel → Restart Kernel and Run All Cells** : le notebook doit passer de haut en bas.
Puis, dans un terminal : `jupyter nbconvert --to html corrige-3b.ipynb`.